<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/05_interpretabilidade_e_conclusoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: Capítulo 5, Interpretabilidade e Conclusões

Continuação dos capítulos 1 a 4. Esse notebook parte do zero de novo (carrega os 9 CSVs, remonta o dataframe mestre e as features), autocontido, mas assume que você já leu os quatro anteriores.

Duas coisas nesse notebook: primeiro, SHAP nos cenários 1 (atraso) e 2 (nota) pra abrir a caixa-preta dos modelos treinados no capítulo 4, e a escolha final de K pro RFM. Segundo, três exportações extras que vão alimentar o capítulo 6 (a vitrine visual final): sweep de threshold, amostra do RFM com cluster, e o fluxo do sankey. Não precisa de T4, roda tudo em CPU. Precisa de `shap` além do que já vínhamos usando.

In [3]:
!pip install shap xgboost

In [4]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

produtos_com_categoria_en = products.merge(category_translation, on='product_category_name', how='left')
mestre = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(produtos_com_categoria_en, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(reviews, on='order_id', how='left')
)
print(f"dataframe mestre: {mestre.shape}")

dataframe mestre: (118310, 40)


## Reconstruindo as features do capítulo 3

In [5]:
geo_media = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

m = mestre.copy()
m['customer_zip_code_prefix'] = m['customer_zip_code_prefix'].astype('int64')
m['seller_zip_code_prefix'] = m['seller_zip_code_prefix'].astype('int64')
geo_media['geolocation_zip_code_prefix'] = geo_media['geolocation_zip_code_prefix'].astype('int64')
m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}), on='customer_zip_code_prefix', how='left')
m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix', 'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}), on='seller_zip_code_prefix', how='left')
m['distance_km'] = haversine(m['cust_lat'], m['cust_lng'], m['sell_lat'], m['sell_lng'])
m['approval_hours'] = (m['order_approved_at'] - m['order_purchase_timestamp']).dt.total_seconds() / 3600
m['month'] = m['order_purchase_timestamp'].dt.month
m['atraso_dias'] = (m['order_delivered_customer_date'] - m['order_estimated_delivery_date']).dt.days
m['delivery_days'] = (m['order_delivered_customer_date'] - m['order_purchase_timestamp']).dt.days
print("features reconstruídas")

features reconstruídas


## SHAP no cenário 1: por que o modelo pensa que vai atrasar

O capítulo 4 já mostrou a importância de feature "global" (`feature_importances_`) do modelo honesto. SHAP vai mais fundo: calcula, pra cada previsão individual, o quanto cada feature empurrou o resultado pra cima ou pra baixo, e a média do valor absoluto disso vira uma importância mais rigorosa que a do capítulo 4. Uso o XGBoost aqui (não o Random Forest), porque foi o único que realmente "aposta" no positivo no capítulo 4, faz mais sentido abrir a caixa-preta de um modelo que decide alguma coisa.

In [6]:
delay_df = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'distance_km', 'product_weight_g', 'price', 'freight_value', 'approval_hours', 'month', 'atraso_dias']]
    .dropna()
)
delay_df['atrasado'] = (delay_df['atraso_dias'] > 0).astype(int)

features_1 = ['distance_km', 'product_weight_g', 'price', 'freight_value', 'approval_hours', 'month']
X1, y1 = delay_df[features_1], delay_df['atrasado']
X1tr, X1te, y1tr, y1te = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)

modelo_delay = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss')
modelo_delay.fit(X1tr, y1tr)

explainer_delay = shap.TreeExplainer(modelo_delay)
shap_values_delay = explainer_delay.shap_values(X1te)
importancia_delay = np.abs(shap_values_delay).mean(axis=0)

shap_delay = sorted(
    [{'feature': f, 'value': round(float(v), 4)} for f, v in zip(features_1, importancia_delay)],
    key=lambda r: -r['value'],
)
with open('shap-delay.json', 'w', encoding='utf-8') as f:
    json.dump(shap_delay, f, ensure_ascii=False, indent=2)
for r in shap_delay:
    print(f"{r['feature']}: {r['value']}")

month: 0.5577
distance_km: 0.2793
freight_value: 0.2224
approval_hours: 0.1894
price: 0.1643
product_weight_g: 0.1581


## SHAP no cenário 2: o que pesa na nota

Mesma lógica, agora pro XGBoost regressor que prevê `review_score`.

In [7]:
s2 = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'atraso_dias', 'delivery_days', 'price', 'freight_value', 'payment_installments', 'review_score']]
    .dropna()
)
features_2 = ['atraso_dias', 'delivery_days', 'price', 'freight_value', 'payment_installments']
X2, y2 = s2[features_2], s2['review_score']
X2tr, X2te, y2tr, y2te = train_test_split(X2, y2, test_size=0.2, random_state=42)

modelo_review = xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
modelo_review.fit(X2tr, y2tr)

explainer_review = shap.TreeExplainer(modelo_review)
shap_values_review = explainer_review.shap_values(X2te)
importancia_review = np.abs(shap_values_review).mean(axis=0)

shap_review = sorted(
    [{'feature': f, 'value': round(float(v), 4)} for f, v in zip(features_2, importancia_review)],
    key=lambda r: -r['value'],
)
with open('shap-review.json', 'w', encoding='utf-8') as f:
    json.dump(shap_review, f, ensure_ascii=False, indent=2)
for r in shap_review:
    print(f"{r['feature']}: {r['value']}")

atraso_dias: 0.2089
delivery_days: 0.1473
freight_value: 0.0708
price: 0.0678
payment_installments: 0.0467


## RFM: escolhendo o K final

O capítulo 4 varreu K de 2 a 8 e achou K=2 disparado na frente em silhueta, mas isso é só o corte trivial (compra única vs. recorrente) que o capítulo 3 já tinha achado sem clustering nenhum. Descartando o K=2 trivial, o próximo melhor por silhueta é **K=4** (0,49, maior que 5, 6, 7 e 8), então é esse que uso pra segmentação de verdade.

In [8]:
data_max = orders['order_purchase_timestamp'].max()
orders_validos = orders[orders['order_status'] != 'canceled']
oc = orders_validos.merge(customers, on='customer_id', how='left')
pagamento_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()
oc = oc.merge(pagamento_por_pedido, on='order_id', how='left')

rfm = oc.groupby('customer_unique_id').agg(
    recencia_dias=('order_purchase_timestamp', lambda x: (data_max - x.max()).days),
    frequencia=('order_id', 'nunique'),
    valor_monetario=('payment_value', 'sum'),
).reset_index().dropna()

scaler = StandardScaler()
rfm_escalado = scaler.fit_transform(rfm[['recencia_dias', 'frequencia', 'valor_monetario']])

kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['cluster'] = kmeans_final.fit_predict(rfm_escalado)

perfil = rfm.groupby('cluster').agg(
    recencia_media=('recencia_dias', 'mean'),
    frequencia_media=('frequencia', 'mean'),
    valor_medio=('valor_monetario', 'mean'),
    n_clientes=('customer_unique_id', 'count'),
).reset_index()

perfil_json = [
    {
        'cluster': int(row.cluster),
        'recencia_media': round(float(row.recencia_media), 1),
        'frequencia_media': round(float(row.frequencia_media), 3),
        'valor_medio': round(float(row.valor_medio), 2),
        'n_clientes': int(row.n_clientes),
    }
    for row in perfil.itertuples()
]
with open('rfm-clusters.json', 'w', encoding='utf-8') as f:
    json.dump(perfil_json, f, ensure_ascii=False, indent=2)
print(perfil)

   cluster  recencia_media  frequencia_media  valor_medio  n_clientes
0        0      177.352839          1.000000   134.581894       51780
1        1      290.387263          1.012658  1160.186934        2528
2        2      437.446478          1.000000   133.715545       38358
3        3      270.374568          2.114029   288.511617        2894


## Preparando o capítulo 6

Três exportações extras que não viram prosa aqui, mas alimentam a vitrine visual do próximo capítulo: uma amostra do RFM com o cluster de cada cliente (pro scatter 3D), o sweep de threshold do cenário de atraso (pro slider interativo), e o fluxo região → entrega → nota (pro sankey).

In [9]:
amostra = rfm.sample(n=min(500, len(rfm)), random_state=42)[['recencia_dias', 'frequencia', 'valor_monetario', 'cluster']]
amostra_json = [
    {
        'recencia_dias': round(float(row.recencia_dias), 1),
        'frequencia': int(row.frequencia),
        'valor_monetario': round(float(row.valor_monetario), 2),
        'cluster': int(row.cluster),
    }
    for row in amostra.itertuples()
]
with open('rfm-cluster-sample.json', 'w', encoding='utf-8') as f:
    json.dump(amostra_json, f, ensure_ascii=False, indent=2)
print(f"{len(amostra_json)} clientes amostrados")

500 clientes amostrados


In [10]:
modelos_sweep = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss'),
}

sweep = []
for nome, modelo in modelos_sweep.items():
    modelo.fit(X1tr, y1tr)
    proba = modelo.predict_proba(X1te)[:, 1]
    for limiar in np.arange(0.0, 1.01, 0.02):
        pred = (proba >= limiar).astype(int)
        sweep.append({
            'model': nome,
            'threshold': round(float(limiar), 2),
            'precision': round(float(precision_score(y1te, pred, zero_division=0)), 4),
            'recall': round(float(recall_score(y1te, pred, zero_division=0)), 4),
            'f1': round(float(f1_score(y1te, pred, zero_division=0)), 4),
        })

with open('threshold-sweep.json', 'w', encoding='utf-8') as f:
    json.dump(sweep, f, ensure_ascii=False, indent=2)
print(f"{len(sweep)} pontos de threshold exportados")

102 pontos de threshold exportados


In [11]:
regiao_por_uf = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}

sk = m.drop_duplicates(subset='order_id')[['order_id', 'customer_state', 'atraso_dias', 'review_score']].dropna()
sk['regiao'] = sk['customer_state'].map(regiao_por_uf)
sk['status_entrega'] = np.where(sk['atraso_dias'] > 0, 'Atrasado', 'No prazo')
sk['faixa_nota'] = pd.cut(sk['review_score'], bins=[0, 2, 3, 5], labels=['Nota baixa (1-2)', 'Nota média (3)', 'Nota alta (4-5)'])

fluxo1 = sk.groupby(['regiao', 'status_entrega']).size().reset_index(name='value')
fluxo2 = sk.groupby(['status_entrega', 'faixa_nota'], observed=True).size().reset_index(name='value')

links = (
    [{'source': r['regiao'], 'target': r['status_entrega'], 'value': int(r['value'])} for _, r in fluxo1.iterrows()]
    + [{'source': r['status_entrega'], 'target': r['faixa_nota'], 'value': int(r['value'])} for _, r in fluxo2.iterrows()]
)
nomes = sorted(set([l['source'] for l in links] + [l['target'] for l in links]))
sankey = {'nodes': [{'name': n} for n in nomes], 'links': links}

with open('sankey-flow.json', 'w', encoding='utf-8') as f:
    json.dump(sankey, f, ensure_ascii=False, indent=2)
print(json.dumps(sankey, ensure_ascii=False, indent=2))

{
  "nodes": [
    {
      "name": "Atrasado"
    },
    {
      "name": "Centro-Oeste"
    },
    {
      "name": "No prazo"
    },
    {
      "name": "Nordeste"
    },
    {
      "name": "Norte"
    },
    {
      "name": "Nota alta (4-5)"
    },
    {
      "name": "Nota baixa (1-2)"
    },
    {
      "name": "Nota média (3)"
    },
    {
      "name": "Sudeste"
    },
    {
      "name": "Sul"
    }
  ],
  "links": [
    {
      "source": "Centro-Oeste",
      "target": "Atrasado",
      "value": 361
    },
    {
      "source": "Centro-Oeste",
      "target": "No prazo",
      "value": 5233
    },
    {
      "source": "Nordeste",
      "target": "Atrasado",
      "value": 1121
    },
    {
      "source": "Nordeste",
      "target": "No prazo",
      "value": 7854
    },
    {
      "source": "Norte",
      "target": "Atrasado",
      "value": 148
    },
    {
      "source": "Norte",
      "target": "No prazo",
      "value": 1631
    },
    {
      "source": "Sudeste",
     